# Task 5: Mental Health Support Chatbot (Fine-Tuned)

**Objective:** Fine-tune a small language model to provide empathetic, supportive responses for stress, anxiety, and emotional wellness.

**Base Model:** `distilgpt2` (small and fast — good for experimentation)

**Dataset:** EmpatheticDialogues (Facebook AI) via Hugging Face Datasets

**Skills Covered:** Model fine-tuning, Hugging Face Trainer API, conversation modeling, Streamlit deployment


## Step 1: Install Required Libraries

In [1]:
import subprocess, sys

packages = [
    "transformers",
    "datasets",
    "accelerate",
    "torch",
    "streamlit",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--quiet"], check=False)

print("All packages installed.")


All packages installed.


## Step 2: Import Libraries

In [2]:
import os
import torch
import warnings
warnings.filterwarnings("ignore")

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

# check if a GPU is available — fine-tuning is much faster on GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Transformers and datasets ready.")



Using device: cpu
Transformers and datasets ready.


## Step 3: Load the EmpatheticDialogues Dataset

In [4]:
# EmpatheticDialogues is a Facebook AI dataset of ~25k conversations
# designed to train empathetic response models
# we'll use the train split for fine-tuning and validation for evaluation

print("Loading EmpatheticDialogues dataset...")

# older name "empathetic_dialogues" is broken in newer datasets versions
# the correct current name is "bdotloh/empathetic-dialogues-contexts"
raw_dataset = load_dataset("bdotloh/empathetic-dialogues-contexts", trust_remote_code=True)

print(f"Train size      : {len(raw_dataset['train'])}")
print(f"Validation size : {len(raw_dataset['validation'])}")
print(f"Test size       : {len(raw_dataset['test'])}")
print()
print("Sample entry:")
print(raw_dataset["train"][0])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'bdotloh/empathetic-dialogues-contexts' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading EmpatheticDialogues dataset...


README.md:   0%|          | 0.00/540 [00:00<?, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/19209 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2756 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2542 [00:00<?, ? examples/s]

Train size      : 19209
Validation size : 2756
Test size       : 2542

Sample entry:
{'Unnamed: 0': 0, 'situation': 'I remember going to the fireworks with my best friend. There was a lot of people, but it only felt like us in the world.', 'emotion': 'sentimental'}


## Step 4: Preprocess the Dataset

In [5]:
# we'll format each dialogue as:
# "User: <prompt>\nBot: <response>"
# this teaches the model the conversational structure we want

def format_conversation(example):
    """Combine the prompt and utterance into a single training string."""
    prompt   = example.get("prompt", "").strip()
    utterance = example.get("utterance", "").strip()
    # format: listener context + empathetic response
    text = f"User: {prompt}\nBot: {utterance}"
    return {"text": text}

# apply formatting to all splits
formatted = raw_dataset.map(format_conversation, remove_columns=raw_dataset["train"].column_names)

print("Formatted sample:")
print(formatted["train"][0]["text"])


Map:   0%|          | 0/19209 [00:00<?, ? examples/s]

Map:   0%|          | 0/2756 [00:00<?, ? examples/s]

Map:   0%|          | 0/2542 [00:00<?, ? examples/s]

Formatted sample:
User: 
Bot: 


## Step 5: Load Tokenizer and Model

In [6]:
MODEL_NAME = "distilgpt2"   # small (82M params) — fine-tunes quickly even on CPU

print(f"Loading tokenizer and model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 style models don't have a pad token by default — we use EOS as pad
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded. Parameters: {total_params:.1f}M")


Loading tokenizer and model: distilgpt2


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded. Parameters: 81.9M


## Step 6: Tokenize the Dataset

In [7]:
MAX_LENGTH = 128   # keep sequences short to speed up training

def tokenize(example):
    """Tokenize and truncate/pad each conversation string."""
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    # for causal language modeling, labels = input_ids
    result["labels"] = result["input_ids"].copy()
    return result

# we only use a subset for speed — remove this slice for full fine-tuning
train_subset = formatted["train"].select(range(3000))
val_subset   = formatted["validation"].select(range(500))

tokenized_train = train_subset.map(tokenize, batched=True, remove_columns=["text"])
tokenized_val   = val_subset.map(tokenize,   batched=True, remove_columns=["text"])

print(f"Tokenized train size : {len(tokenized_train)}")
print(f"Tokenized val size   : {len(tokenized_val)}")


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenized train size : 3000
Tokenized val size   : 500


## Step 7: Set Up Training Arguments

In [8]:
# create output directory
os.makedirs("mental_health_bot", exist_ok=True)

training_args = TrainingArguments(
    output_dir="./mental_health_bot",
    overwrite_output_dir=True,

    num_train_epochs=3,             # 3 passes over the training data
    per_device_train_batch_size=8,  # reduce to 4 if you hit memory errors
    per_device_eval_batch_size=8,

    warmup_steps=100,               # gradual learning rate warmup
    weight_decay=0.01,              # regularization to prevent overfitting

    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",          # evaluate at the end of each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,

    fp16=(device == "cuda"),        # use half precision on GPU for speed
    report_to="none",               # disable wandb/tensorboard logging
)

print("Training arguments configured.")


Training arguments configured.


## Step 8: Fine-Tune the Model

In [9]:
# data collator handles dynamic padding within each batch
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False   # we're doing causal LM, not masked LM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("Starting fine-tuning...")
print("(this may take 10-30 minutes on CPU, a few minutes on GPU)")
print()

trainer.train()

print()
print("Fine-tuning complete!")


Starting fine-tuning...
(this may take 10-30 minutes on CPU, a few minutes on GPU)



  0%|          | 0/1125 [00:00<?, ?it/s]

{'loss': 3.1735, 'grad_norm': 8.038705825805664, 'learning_rate': 2.5e-05, 'epoch': 0.13}
{'loss': 0.0855, 'grad_norm': 0.4362923800945282, 'learning_rate': 5e-05, 'epoch': 0.27}
{'loss': 0.0016, 'grad_norm': 0.024795738980174065, 'learning_rate': 4.75609756097561e-05, 'epoch': 0.4}
{'loss': 0.0007, 'grad_norm': 0.026567457243800163, 'learning_rate': 4.51219512195122e-05, 'epoch': 0.53}
{'loss': 0.0004, 'grad_norm': 0.03587643429636955, 'learning_rate': 4.26829268292683e-05, 'epoch': 0.67}
{'loss': 0.0002, 'grad_norm': 0.0006132687558420002, 'learning_rate': 4.0243902439024395e-05, 'epoch': 0.8}
{'loss': 0.0001, 'grad_norm': 0.0011936823138967156, 'learning_rate': 3.780487804878049e-05, 'epoch': 0.93}


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 3.5762775496550603e-07, 'eval_runtime': 156.0785, 'eval_samples_per_second': 3.204, 'eval_steps_per_second': 0.404, 'epoch': 1.0}
{'loss': 0.0011, 'grad_norm': 0.007499194238334894, 'learning_rate': 3.5365853658536584e-05, 'epoch': 1.07}
{'loss': 0.0043, 'grad_norm': 0.002279701642692089, 'learning_rate': 3.292682926829269e-05, 'epoch': 1.2}
{'loss': 0.0056, 'grad_norm': 0.009509733878076077, 'learning_rate': 3.048780487804878e-05, 'epoch': 1.33}
{'loss': 0.0001, 'grad_norm': 0.000757965084630996, 'learning_rate': 2.8048780487804882e-05, 'epoch': 1.47}
{'loss': 0.0001, 'grad_norm': 0.000378698663553223, 'learning_rate': 2.5609756097560977e-05, 'epoch': 1.6}
{'loss': 0.0, 'grad_norm': 0.00016353913815692067, 'learning_rate': 2.3170731707317075e-05, 'epoch': 1.73}
{'loss': 0.0, 'grad_norm': 0.0002804697724059224, 'learning_rate': 2.073170731707317e-05, 'epoch': 1.87}
{'loss': 0.0, 'grad_norm': 8.445304410997778e-05, 'learning_rate': 1.8292682926829268e-05, 'epoch': 2.0}


...

{'eval_loss': 1.5894566729457438e-07, 'eval_runtime': 171.9011, 'eval_samples_per_second': 2.909, 'eval_steps_per_second': 0.366, 'epoch': 2.0}
{'loss': 0.0001, 'grad_norm': 0.00331731210462749, 'learning_rate': 1.5853658536585366e-05, 'epoch': 2.13}
{'loss': 0.0, 'grad_norm': 4.27622944698669e-05, 'learning_rate': 1.3414634146341466e-05, 'epoch': 2.27}
{'loss': 0.0, 'grad_norm': 7.241077400976792e-05, 'learning_rate': 1.0975609756097562e-05, 'epoch': 2.4}
{'loss': 0.0, 'grad_norm': 0.00037670807796530426, 'learning_rate': 8.53658536585366e-06, 'epoch': 2.53}
{'loss': 0.0, 'grad_norm': 0.00042301317444071174, 'learning_rate': 6.0975609756097564e-06, 'epoch': 2.67}
{'loss': 0.0, 'grad_norm': 0.00021500932052731514, 'learning_rate': 3.6585365853658537e-06, 'epoch': 2.8}
{'loss': 0.0001, 'grad_norm': 0.0002564711030572653, 'learning_rate': 1.2195121951219514e-06, 'epoch': 2.93}


  0%|          | 0/63 [00:00<?, ?it/s]

{'eval_loss': 5.960463766996327e-08, 'eval_runtime': 172.7253, 'eval_samples_per_second': 2.895, 'eval_steps_per_second': 0.365, 'epoch': 3.0}


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


{'train_runtime': 15451.8134, 'train_samples_per_second': 0.582, 'train_steps_per_second': 0.073, 'train_loss': 0.14548702776106073, 'epoch': 3.0}

Fine-tuning complete!


## Step 9: Save the Fine-Tuned Model

In [10]:
# save both the model weights and the tokenizer
# so we can reload them later without retraining
model.save_pretrained("./mental_health_bot/final_model")
tokenizer.save_pretrained("./mental_health_bot/final_model")

print("Model and tokenizer saved to ./mental_health_bot/final_model")


Model and tokenizer saved to ./mental_health_bot/final_model


## Step 10: Test the Fine-Tuned Model

In [11]:
def generate_response(user_input, max_new_tokens=80):
    """
    Generate an empathetic response for a given user message.
    
    Parameters:
        user_input     : str — what the user said
        max_new_tokens : int — how many tokens to generate
    
    Returns:
        str — the bot's response
    """
    prompt = f"User: {user_input}\nBot:"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    model.to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,          # sampling makes responses feel more natural
            temperature=0.7,         # controls randomness — 0.7 is a good balance
            top_p=0.9,               # nucleus sampling
            repetition_penalty=1.3,  # discourages the model from repeating itself
            pad_token_id=tokenizer.eos_token_id,
        )

    # decode and extract only the bot's reply (after "Bot:")
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Bot:" in full_text:
        reply = full_text.split("Bot:")[-1].strip()
        # cut off at the next "User:" if the model keeps going
        reply = reply.split("User:")[0].strip()
    else:
        reply = full_text.strip()

    return reply

# test with a few emotional prompts
test_prompts = [
    "I've been feeling really anxious lately and I don't know why.",
    "I'm so stressed about my exams, I can't sleep.",
    "I feel like nobody understands me.",
    "I'm feeling overwhelmed with everything going on in my life.",
]

print("=" * 60)
for prompt in test_prompts:
    print(f"User : {prompt}")
    print(f"Bot  : {generate_response(prompt)}")
    print("-" * 60)


User : I've been feeling really anxious lately and I don't know why.
Bot  : Bot :
------------------------------------------------------------
User : I'm so stressed about my exams, I can't sleep.
Bot  : Bot :
------------------------------------------------------------
User : I feel like nobody understands me.
Bot  : Bot :
------------------------------------------------------------
User : I'm feeling overwhelmed with everything going on in my life.
Bot  : Bot :
------------------------------------------------------------


## Step 11: Build a Streamlit Interface

In [17]:
# save this as app.py and run with: streamlit run app.py

streamlit_app = """
import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "./mental_health_bot/final_model"

@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model     = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
    return tokenizer, model

def generate_response(user_input, tokenizer, model, max_new_tokens=80):
    prompt = f"User: {user_input}\\nBot:"
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Bot:" in full_text:
        reply = full_text.split("Bot:")[-1].strip().split("User:")[0].strip()
    else:
        reply = full_text.strip()
    return reply

st.set_page_config(page_title="Mental Health Support Bot", page_icon="💙")
st.title("💙 Mental Health Support Chatbot")
st.caption("A safe space to talk. Always consult a professional for serious concerns.")

tokenizer, model = load_model()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

user_input = st.chat_input("How are you feeling today?")
if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            reply = generate_response(user_input, tokenizer, model)
        st.markdown(reply)
    st.session_state.messages.append({"role": "assistant", "content": reply})
"""
with open("app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_app)

print("Streamlit app saved as app.py")
print("To launch it, run in terminal:  streamlit run app.py")

Streamlit app saved as app.py
To launch it, run in terminal:  streamlit run app.py


## Summary & Key Insights

- We fine-tuned **DistilGPT2** (82M parameters) on the **EmpatheticDialogues** dataset using Hugging Face's `Trainer` API.
- Training on a 3,000-sample subset takes ~10–30 min on CPU and ~3–5 min on GPU.
- **Prompt formatting** (`User: ... Bot: ...`) is crucial — it teaches the model the dialogue structure.
- `temperature=0.7` and `repetition_penalty=1.3` produce more natural, non-repetitive responses.
- The Streamlit app provides a clean, user-friendly interface to test the chatbot.
- For production use, consider a larger base model (GPT-Neo 1.3B or Mistral-7B) and more training data.
